**EEG–Eye-Tracking Multimodal Research | Presented by: Reza Saadatyar (2026)**

**06 | EEG–Eye-Tracking Model Selection**

![Ten-model EEG–eye-tracking selection and final Test evidence](../../images/06_robustness_and_final_evaluation.svg)

**Objective:** Select the primary NR/TSR classifier using participant-disjoint Validation, then evaluate its frozen configuration on the independent Test participants. All ten candidates are scored on the same Test windows for descriptive comparison, without Test-based re-selection.

**Evidence:** The 18-person study contains **371 Train, 99 Validation and 160 Test windows**. Ten models provided **990 Validation** and **1,600 Test predictions**, with 11/3/4 independent people in the respective partitions.

**Preselected model:** **Gaze Random Forest** was chosen using Validation Macro F1 **0.6244**. Its final four-person Test performance is **Macro F1 0.4997**, **Balanced Accuracy 0.5250** and **Accuracy 0.5250**. The selected model was not changed after Test scoring.

**Scope:** This is within-source task classification, not direct comprehension measurement. The four Test people provide limited support for wider generalisation.

**00. Colab Repository Setup**

In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")

if not repo_path.exists():
    subprocess.run(
        ["git", "clone", "-q", repo_url, str(repo_path)], check=True
    )

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


/content/Eye-Tracking-Neurocognitive-Intelligence


**01. Imports and Research Environment**

In [2]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eeg_eye_tracking_fusion")

# ---------------------------- 2. Standard Library ----------------------------
import importlib.util
import json
import random
import shutil
import sys

# ---------------------- 3. Colab Dependency Preparation ----------------------
if importlib.util.find_spec("mne") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install",
         "mne>=1.6,<2"],
        check=True,
    )

# ----------------- 4. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 5. Machine Learning and Deep Learning -------------------
import joblib
from sklearn.metrics import ConfusionMatrixDisplay
import torch

# ------------------- 6. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style

# ----------------- 7. Shared and Project Research Utilities ------------------
from core.output_download import download_outputs_from_colab
from features import verify_temporal_feature_alignment
from models import (
    MODEL_ORDER, CLASSICAL_ORDER, NEURAL_ORDER, CLASS_TO_INDEX,
    metrics, matrix, probabilities, record_predictions,
    torch_network, predict_neural, feature_columns,
    participant_prediction_metrics,
)

# ---------------------------- 8. Reproducibility -----------------------------
SEED = 17
random.seed(SEED)
np.random.seed(SEED)

# ------------------- 9. Output Directory and Setup Summary -------------------
work = root / "eeg_eye_tracking_fusion"
output_dir = work / "output"
output_dir.mkdir(parents=True, exist_ok=True)
raw_dir = Path("/content/zuco2_source_cache")
raw_dir.mkdir(parents=True, exist_ok=True)
plot_style(font_family="DejaVu Sans", figure_dpi=100)
print("Project outputs:", output_dir)
print("Temporary source cache:", raw_dir)
print("Python:", sys.version.split()[0])
print("Free disk (GiB):", round(
    shutil.disk_usage("/content").free / 2**30, 2,
))


Local changes kept; continuing with the current checkout. Conflicting Git update deferred.
Project outputs: /content/Eye-Tracking-Neurocognitive-Intelligence/eeg_eye_tracking_fusion/output
Temporary source cache: /content/zuco2_source_cache
Python: 3.13.18
Free disk (GiB): 103.96


**02. Reconcile Predictions Across Modelling Stages**

The comparison brings together **693 classical** and **297 GRU** prediction records. Each of the **ten** candidates must cover the same **99 original Validation window IDs** with matching participant IDs, NR/TSR labels and class-probability order.

| Reconciliation check | Observed result |
| --- | --- |
| Candidate models | **10** |
| Model/window prediction records | **990**, all uniquely identified |
| Distinct Validation windows | **99** from **3 people** |
| Target classes | **60 NR / 39 TSR** |
| Stage-05 scaling and checkpoints | Consistent with Train-derived values and saved architectures |

All ten pooled metrics were recalculated from their saved predictions and matched the recorded values. These checks show that stored model outputs can be compared on the same Validation windows; they do not constitute a new training run.

In [3]:
# ========= 02. Load Upstream Models and Reconcile Validation Records =========

# --------------------- 1. Source Role and Identity Audit ---------------------
table = pd.read_csv(
    output_dir / '03_features.csv', dtype={'participant_id': str})
registry = pd.read_csv(
    output_dir / '02_window_registry.csv',
    dtype={'participant_id': str})
if table.empty or table.window_id.duplicated().any():
    raise ValueError('Missing or duplicated Stage-03 feature identities')
if not table.split.isin(('train', 'validation', 'test')).all():
    raise ValueError('Unexpected Stage-03 split role')
if not table.task.isin(CLASS_TO_INDEX).all():
    raise ValueError('Unexpected Stage-03 task label')
eligible = registry.loc[
    registry.eligible.eq(True),
    ['window_id', 'participant_id', 'task', 'split']]
if eligible.window_id.duplicated().any():
    raise ValueError('Duplicated eligible Stage-02 window identities')
if set(table.window_id) != set(eligible.window_id):
    raise ValueError('Stage-02 and Stage-03 window identities differ')
matched = table[
    ['window_id', 'participant_id', 'task', 'split']
].merge(
    eligible, on='window_id', validate='one_to_one',
    suffixes=('_03', '_02'))
for field in ('participant_id', 'task', 'split'):
    if not matched[f'{field}_03'].eq(matched[f'{field}_02']).all():
        raise ValueError(f'Stage-02/03 {field} mismatch')
if not table.groupby('participant_id').split.nunique().eq(1).all():
    raise ValueError('Participant overlap between Train, Validation and Test')

# The current six-person pilot metrics cannot be mixed with a new
# Stage-01/02/03 cohort. Test roles are audited but no Test labels scored.
protocol = json.loads((output_dir / '01_protocol.json').read_text())
if (protocol.get('selected_people') != table.participant_id.nunique()
        or protocol.get('target') != CLASS_TO_INDEX
        or set(table.participant_id) !=
        set(pd.read_csv(
            output_dir / '01_split_assignments.csv',
            dtype={'participant_id': str}).participant_id)):
    raise ValueError('Stage-01/02/03 cohort identities or labels differ')
locked = protocol.get('pilot_role_lock')
if not isinstance(locked, dict):
    raise ValueError('Expanded cohort is missing locked pilot role provenance')
for person, role in locked.items():
    actual = table.loc[table.participant_id.eq(person), 'split'].unique()
    if len(actual) != 1 or actual[0] != role:
        raise ValueError(f'Pilot participant role changed: {person}')

valid = table.loc[table.split.eq('validation')].copy()
y_valid = valid.task.map(CLASS_TO_INDEX).to_numpy(dtype=int)
test_n_people = table.loc[
    table.split.eq('test'), 'participant_id'].nunique()
if (valid.empty or valid.task.nunique() != 2 or
        valid.participant_id.nunique() < 1 or test_n_people < 1):
    raise ValueError('Missing Validation classes or reserved Test roles')

# --------------- 2. Temporal Inputs and Train-Fitted Scaler QA ---------------
# Check only Train/Validation sequence rows here; do not score Test.
dev_idx = np.where(table.split.ne('test').to_numpy())[0]
train_idx = np.where(table.split.eq('train').to_numpy())[0]
if len(train_idx) < 1:
    raise ValueError('Training partition is empty')
with np.load(output_dir / '03_temporal_sequences.npz',
             allow_pickle=False) as temporal:
    development = {
        key: temporal[key][dev_idx] for key in ('eeg', 'gaze')
    }
    training = {
        key: temporal[key][train_idx] for key in ('eeg', 'gaze')
    }
verify_temporal_feature_alignment(
    development, table.iloc[dev_idx].reset_index(drop=True))
with np.load(output_dir / '05_scaling.npz',
             allow_pickle=False) as recorded_scale:
    for key, sequence in training.items():
        flat = sequence.reshape(-1, sequence.shape[-1])
        median = np.nanmedian(flat, axis=0)
        median = np.where(np.isfinite(median), median, 0.)
        filled = np.where(np.isfinite(flat), flat, median)
        expected_scale = {
            'median': median.astype('float32'),
            'mean': filled.mean(axis=0).astype('float32'),
            'std': np.maximum(
                filled.std(axis=0), 1e-6).astype('float32'),
        }
        for name, expected in expected_scale.items():
            recorded = recorded_scale[f'{key}_{name}']
            if (recorded.shape != expected.shape or
                    not np.isfinite(recorded).all() or
                    not np.allclose(
                        recorded, expected, rtol=1e-5, atol=1e-5)):
                raise ValueError(
                    f'Stage-05 {key}_{name} disagrees with '
                    'Train-derived Stage-03 sequences')
print('Train/Validation temporal rows and Stage-05 scaling verified')

# ---------------------- 3. Candidate Table Consistency -----------------------
source = pd.concat([
    pd.read_csv(output_dir / '04_validation_predictions.csv',
                dtype={'participant_id': str}),
    pd.read_csv(output_dir / '05_validation_predictions.csv',
                dtype={'participant_id': str})],
    ignore_index=True)
summaries = pd.concat([
    pd.read_csv(output_dir / '04_validation_metrics.csv'),
    pd.read_csv(output_dir / '05_validation_metrics.csv')],
    ignore_index=True)
if (len(summaries) != len(MODEL_ORDER) or
        summaries.model.duplicated().any() or
        set(summaries.model) != set(MODEL_ORDER) or
        set(source.model) != set(MODEL_ORDER)):
    raise ValueError('Missing, duplicated or unexpected model candidates')
if not source.split.eq('validation').all():
    raise ValueError('Non-Validation rows in the candidate predictions')
if not summaries.partition.eq('validation').all():
    raise ValueError('Non-Validation rows in candidate metrics')
if source.duplicated(['model', 'window_id']).any():
    raise ValueError('Duplicate model/window prediction records')

# --------------------- 4. Saved Temporal Checkpoints QA ----------------------
# Inspect checkpoint metadata and state shapes, without scoring any data.
for name in NEURAL_ORDER:
    checkpoint_path = output_dir / f'05_{name}.pt'
    checkpoint = torch.load(
        checkpoint_path, map_location='cpu', weights_only=True)
    stored = summaries.loc[summaries.model.eq(name)].iloc[0]
    if (checkpoint.get('model') != name or
            int(checkpoint['best_epoch']) != int(stored.best_epoch)):
        raise ValueError(f'Stage-05 checkpoint metadata mismatch: {name}')
    network = torch_network(
        name, development['eeg'].shape[-1],
        development['gaze'].shape[-1],
        hidden=int(checkpoint['hidden']))
    network.load_state_dict(checkpoint['state_dict'], strict=True)
    count = sum(param.numel() for param in network.parameters())
    if count != int(stored.parameters):
        raise ValueError(f'Stage-05 parameter count mismatch: {name}')
print('Three saved temporal checkpoints match current model definitions')

# ------------------------ 5. Aligned Prediction Audit ------------------------
for name in MODEL_ORDER:
    part = source.loc[source.model.eq(name)]
    if len(part) != len(valid) or set(part.window_id) != set(valid.window_id):
        raise ValueError(f'Validation window IDs mismatch: {name}')
    aligned = part.set_index('window_id').loc[valid.window_id]
    for field in ('participant_id', 'task', 'split'):
        if not np.array_equal(
                aligned[field].to_numpy(),
                valid[field].to_numpy()):
            raise ValueError(f'Validation {field} mismatch: {name}')
    if not np.array_equal(
            aligned.target_index.to_numpy(dtype=int), y_valid):
        raise ValueError(f'Validation true labels differ: {name}')
    proba = aligned[['p_nr', 'p_tsr']].to_numpy(dtype=float)
    if (not np.isfinite(proba).all() or
            (proba < 0).any() or (proba > 1).any() or
            not np.allclose(proba.sum(axis=1), 1, atol=1e-5) or
            not np.array_equal(aligned.pred_index.to_numpy(dtype=int),
                               proba.argmax(axis=1))):
        raise ValueError(f'Invalid Validation probabilities: {name}')
    recorded = summaries.loc[summaries.model.eq(name)].iloc[0]
    if (int(recorded.n_windows) != len(valid) or
            int(recorded.n_people) != valid.participant_id.nunique()):
        raise ValueError(f'Validation support mismatch: {name}')
    recomputed = metrics(
        y_valid, aligned.pred_index.to_numpy(dtype=int))
    for metric, score in recomputed.items():
        if not np.isclose(score, recorded[metric], atol=1e-9):
            raise ValueError(f'Saved {metric} differs: {name}')
print('Reconciled', len(MODEL_ORDER), 'models on', len(valid),
      'Validation windows from', valid.participant_id.nunique(),
      'independent person(s). Test remains unscored.')


Train/Validation temporal rows and Stage-05 scaling verified
Three saved temporal checkpoints match current model definitions
Reconciled 10 models on 99 Validation windows from 3 independent person(s). Test remains unscored.


**03. Select the Primary Development Model**

Candidates are ranked by **Validation Macro F1**, then Balanced Accuracy and, for an exact remaining tie, a predefined priority order. This emphasises both NR and TSR rather than favouring the more frequent class.

| Rank | Model | Macro F1 | Balanced Accuracy |
| ---: | --- | ---: | ---: |
| 1 | **Gaze Random Forest** | **0.6244** | **0.6231** |
| 2 | Gated Fusion GRU | 0.6107 | 0.6096 |
| 3 | Gaze GRU | 0.5938 | 0.5929 |
| 4 | EEG GRU | 0.5938 | 0.5929 |
| 5 | Late Equal Fusion | 0.5643 | 0.5641 |
| 6 | Gaze Logistic | 0.5472 | 0.5474 |
| 7 | EEG Random Forest | 0.5139 | 0.5282 |
| 8 | EEG Logistic | 0.4545 | 0.4737 |
| 9 | Early Fusion Logistic | 0.4518 | 0.4603 |
| 10 | Class Prior | 0.3774 | 0.5000 |

Gaze Random Forest correctly classifies **44/60 NR** and **20/39 TSR** windows. Its individual Macro F1 values (**0.7654**, **0.6306**, **0.3478**) yield an unweighted person-mean of **0.5813**, compared with pooled-window Macro F1 **0.6244**. The distinction demonstrates why the number of **independent people** matters alongside the number of windows.

In [4]:
# =========== 03. Freeze the Validation-Selected Primary Candidate ============

# -------------------------- 1. Primary Ranking Rule --------------------------
ranking = summaries[['model', 'macro_f1', 'balanced_accuracy']].copy()
ranking['priority'] = ranking.model.map(MODEL_ORDER.index)
ranking = ranking.sort_values(
    ['macro_f1', 'balanced_accuracy', 'priority'],
    ascending=[False, False, True]).reset_index(drop=True)
selected_name = str(ranking.iloc[0].model)
protocol = json.loads((output_dir / '01_protocol.json').read_text())
if protocol.get('target') != CLASS_TO_INDEX:
    raise ValueError('Source protocol labels differ from fixed NR/TSR order')

# --------------------------- 2. Test Policy Guard ----------------------------
RUN_FINAL_TEST = True   # Execute the frozen, participant-disjoint final Test.
evaluation_mode = 'final' if RUN_FINAL_TEST else 'development'
if not RUN_FINAL_TEST:
    prior_test_files = [
        name for name in (
            '06_test_metrics.csv', '06_test_predictions.csv',
            '06_person_metrics.csv', '06_cluster_bootstrap.csv',
            '06_primary_test_confusion.png'
        ) if (output_dir / name).exists()
    ]
    if prior_test_files:
        raise RuntimeError(
            'Previous Test artefacts exist in this checkout. '
            'Use a clean development checkout; existing evidence was '
            'preserved and no Test results were opened: '
            + ', '.join(prior_test_files)
        )

# --------------------- 3. Save Validation-Only Decision ----------------------
decision = pd.DataFrame([{
    'selected_model': selected_name,
    'validation_macro_f1': float(ranking.iloc[0].macro_f1),
    'validation_n_people': valid.participant_id.nunique(),
    'test_used_in_selection': False,
    'run_mode': evaluation_mode,
    'source_people': table.participant_id.nunique(),
    'dataset_version': protocol['dataset_version'],
}])
decision.to_csv(output_dir / '06_model_selection.csv', index=False)
# Descriptive person-balanced view supplements, but never replaces, the
# preregistered pooled-window Validation selection criterion.
person_scores = participant_prediction_metrics(source)
person_scores.to_csv(
    output_dir / '06_validation_person_metrics.csv', index=False)
person_summary = (person_scores.groupby('model')
                  .agg(n_people=('participant_id', 'nunique'),
                       mean_person_macro_f1=('macro_f1', 'mean'),
                       minimum_person_macro_f1=('macro_f1', 'min'))
                  .reset_index())
print('Validation participant-level summary (descriptive only):')
display(person_summary.sort_values(
    'mean_person_macro_f1', ascending=False))
display(ranking)
print('Frozen Validation primary:', selected_name)
print('Test scoring enabled:', RUN_FINAL_TEST)


Validation participant-level summary (descriptive only):


,model,n_people,mean_person_macro_f1,minimum_person_macro_f1
6,gaze_forest,3,0.581262,0.347826
5,gated_fusion_gru,3,0.465441,0.392157
7,gaze_gru,3,0.441488,0.392157
3,eeg_gru,3,0.441488,0.392157
8,gaze_logistic,3,0.415808,0.302326
2,eeg_forest,3,0.380398,0.321429
0,class_prior,3,0.378995,0.344828
9,late_equal_fusion,3,0.371195,0.321429
4,eeg_logistic,3,0.338016,0.193548
1,early_fusion_logistic,3,0.308852,0.205128


,model,macro_f1,balanced_accuracy,priority
0,gaze_forest,0.624390,0.623077,4
1,gated_fusion_gru,0.610718,0.609615,9
2,gaze_gru,0.593767,0.592949,7
3,eeg_gru,0.593767,0.592949,8
4,late_equal_fusion,0.564345,0.564103,6
5,gaze_logistic,0.547176,0.547436,1
6,eeg_forest,0.513912,0.528205,5
7,eeg_logistic,0.454490,0.473718,2
8,early_fusion_logistic,0.451805,0.460256,3
9,class_prior,0.377358,0.500000,0


Frozen Validation primary: gaze_forest
Test scoring enabled: True


**04. Evaluate the Frozen Candidate on the Held-Out Test**

The four Test participants (**160 windows, 80 NR and 80 TSR**) were excluded from fitting, checkpoint selection and the ten-model Validation ranking. **Gaze Random Forest** was selected before Test labels were scored.

The final execution used **`RUN_FINAL_TEST=True`**. The saved `06_model_selection.csv` retains `selected_model=gaze_forest`, `test_used_in_selection=False` and `run_mode=final`. All ten previously fitted models were scored on the same Test windows as a secondary comparison; their Test rankings did **not** replace the Validation-selected primary.

**Primary Test result:** Accuracy **0.5250**, Macro F1 **0.4997** and Balanced Accuracy **0.5250**. The confusion matrix records **60/80 NR** and **24/80 TSR** windows classified correctly.

In [5]:
# ========= 04. Score Frozen Models on the Test Only When Authorised ==========

test_predictions, test_scores = [], []
if RUN_FINAL_TEST:
    # Test rows and labels are only accessed inside the authorised branch.
    test = table.loc[table.split.eq('test')].copy()
    y_test = test.task.map(CLASS_TO_INDEX).to_numpy()
    scaling = np.load(output_dir / '05_scaling.npz',
                      allow_pickle=False)
    sequences = np.load(output_dir / '03_temporal_sequences.npz',
                        allow_pickle=False)
    verify_temporal_feature_alignment(sequences, table)
    test_idx = np.where(table.split.eq('test').to_numpy())[0]

    def standardised(key):
        original = sequences[key]
        median = scaling[f'{key}_median']
        mean = scaling[f'{key}_mean']
        std = scaling[f'{key}_std']
        return ((np.where(np.isfinite(original), original, median)
                 - mean) / std).astype('float32')

    eeg_test = standardised('eeg')[test_idx]
    gaze_test = standardised('gaze')[test_idx]
    probas = {}
    for name in CLASSICAL_ORDER:
        if name == 'late_equal_fusion':
            continue
        group = ('eeg' if name.startswith('eeg_') else 'gaze'
                 if name.startswith('gaze_') or name == 'class_prior'
                 else 'fused')
        fitted = joblib.load(output_dir / f'04_{name}.joblib')
        probas[name] = probabilities(fitted, matrix(test, group))
    probas['late_equal_fusion'] = 0.5 * (
        probas['eeg_logistic'] + probas['gaze_logistic'])
    for name in NEURAL_ORDER:
        checkpoint = torch.load(output_dir / f'05_{name}.pt',
                                map_location='cpu',
                                weights_only=True)
        net = torch_network(name, eeg_test.shape[-1],
                            gaze_test.shape[-1],
                            hidden=int(checkpoint['hidden']))
        net.load_state_dict(checkpoint['state_dict'])
        probas[name] = predict_neural(net, eeg_test, gaze_test)
    for name in MODEL_ORDER:
        score = metrics(y_test, probas[name].argmax(axis=1))
        test_scores.append({
            'model': name, 'n_windows': len(test),
            'n_people': test.participant_id.nunique(),
            'selected_primary': name == selected_name, **score})
        test_predictions.append(record_predictions(test, name,
                                                   probas[name]))
    test_metrics = pd.DataFrame(test_scores)
    prediction_table = pd.concat(test_predictions, ignore_index=True)
    test_metrics.to_csv(output_dir / '06_test_metrics.csv',
                        index=False)
    prediction_table.to_csv(output_dir / '06_test_predictions.csv',
                            index=False)
    display(test_metrics.sort_values('macro_f1', ascending=False))
else:
    print('Development mode: no Test scoring performed.')


,model,n_windows,n_people,selected_primary,accuracy,macro_f1,balanced_accuracy
6,late_equal_fusion,160,4,False,0.57500,0.520367,0.57500
9,gated_fusion_gru,160,4,False,0.57500,0.515670,0.57500
3,early_fusion_logistic,160,4,False,0.56250,0.501425,0.56250
4,gaze_forest,160,4,True,0.52500,0.499671,0.52500
7,gaze_gru,160,4,False,0.53125,0.490986,0.53125
1,gaze_logistic,160,4,False,0.51250,0.476422,0.51250
5,eeg_forest,160,4,False,0.55000,0.476364,0.55000
2,eeg_logistic,160,4,False,0.48125,0.386066,0.48125
0,class_prior,160,4,False,0.50000,0.333333,0.50000
8,eeg_gru,160,4,False,0.50000,0.333333,0.50000


**05. Assess Person-Level Test Results and Uncertainty**

The **40 Test model/person rows** describe performance for ten models on four independent people, each contributing 40 paired windows (20 NR and 20 TSR).

| Test participant | Windows | Selected Gaze Forest Macro F1 |
| --- | ---: | ---: |
| `sub-YAC` | 40 | 0.5238 |
| `sub-YAG` | 40 | 0.3333 |
| `sub-YMD` | 40 | 0.4505 |
| `sub-YRP` | 40 | 0.3333 |

A **500-resample participant-cluster bootstrap** for the frozen primary yields a descriptive 95% percentile interval for pooled Macro F1 of **0.3333–0.5616**. With only four independent Test people, this interval is coarse and should not be presented as precise population uncertainty. Individual windows are not independent sampling units.

In [6]:
# ======= 05. Audit Person-Level Test Support and Clustered Uncertainty =======

if RUN_FINAL_TEST:
    groupwise = []
    for name, part in prediction_table.groupby('model'):
        for person, records in part.groupby('participant_id'):
            groupwise.append({'model': name,
                              'participant_id': person,
                              'n_windows': len(records),
                              **metrics(records.target_index,
                                        records.pred_index)})
    grouped = pd.DataFrame(groupwise)
    grouped.to_csv(output_dir / '06_person_metrics.csv', index=False)
    primary = prediction_table.loc[
        prediction_table.model.eq(selected_name)].copy()
    people = primary.participant_id.unique()
    ci = []
    if len(people) >= 3:
        rng = np.random.default_rng(SEED)
        for repeat in range(500):
            sampled = rng.choice(people, len(people), replace=True)
            samples = pd.concat([
                primary.loc[primary.participant_id.eq(p)]
                for p in sampled], ignore_index=True)
            ci.append({'bootstrap_id': repeat,
                       **metrics(samples.target_index,
                                 samples.pred_index)})
    uncertainty = pd.DataFrame(ci, columns=[
        'bootstrap_id', 'accuracy', 'macro_f1',
        'balanced_accuracy'])
    uncertainty.to_csv(output_dir / '06_cluster_bootstrap.csv',
                       index=False)
    if not uncertainty.empty:
        print('Cluster bootstrap Macro F1 95% interval:',
              np.quantile(uncertainty.macro_f1, [0.025, 0.975]))
    else:
        print('Too few independent Test people for bootstrap CI.')
else:
    print('No groupwise Test analysis in development mode.')


Cluster bootstrap Macro F1 95% interval: [0.33333333 0.56157318]


**06. Examine Fixed-Model Input Sensitivity**

The fitted **early-fusion logistic regression** is evaluated on the same **99 Validation windows** with original EEG and gaze features, with EEG features set to missing, and with gaze features set to missing. Missing values are replaced using the **already fitted Train medians**; the model is not retrained.

| Input condition | Validation Macro F1 | Balanced Accuracy |
| --- | ---: | ---: |
| EEG + gaze unchanged | 0.4518 | 0.4603 |
| EEG masked | **0.5343** | **0.5571** |
| Gaze masked | 0.4907 | 0.5385 |

For this fixed classifier, either mask increases Validation Macro F1. Because missing features are replaced using Train medians, the result describes input sensitivity, not a causal estimate of either modality's importance.

In [7]:
# ======== 06. Inspect Validation Sensitivity Without Retuning Models =========

model = joblib.load(
    output_dir / '04_early_fusion_logistic.joblib')
columns = feature_columns(valid, 'fused')
original = valid[columns].copy()
observations = []
for change in ('unchanged', 'mask_eeg', 'mask_gaze'):
    x = original.copy()
    if change != 'unchanged':
        prefix = 'eeg_' if change == 'mask_eeg' else 'gaze_'
        x.loc[:, [col for col in columns if col.startswith(prefix)]] = np.nan
    prediction = probabilities(model, x.to_numpy(dtype=float))
    observations.append({
        'condition': change, 'model': 'early_fusion_logistic',
        'partition': 'validation', 'n_windows': len(valid),
        'n_people': valid.participant_id.nunique(),
        **metrics(y_valid, prediction.argmax(axis=1))})
robustness = pd.DataFrame(observations)
robustness.to_csv(output_dir / '06_validation_sensitivity.csv',
                  index=False)
display(robustness)


,condition,model,partition,n_windows,n_people,accuracy,macro_f1,balanced_accuracy
0,unchanged,early_fusion_logistic,validation,99,3,0.454545,0.451805,0.460256
1,mask_eeg,early_fusion_logistic,validation,99,3,0.626263,0.534266,0.557051
2,mask_gaze,early_fusion_logistic,validation,99,3,0.494949,0.490741,0.538462


**07. Save the Development and Final Test Evidence**

Stage 06 owns **nine outputs**. The four development files remain:

- `06_model_selection.csv`: frozen **Gaze Random Forest** decision, now recorded in final execution mode with Test excluded from selection.
- `06_validation_person_metrics.csv`: 30 candidate/person Validation rows.
- `06_validation_sensitivity.csv`: three fixed early-fusion input conditions.
- `06_validation_evidence.png`: Validation ranking and masking plot.

The five held-out Test outputs are:

- `06_test_metrics.csv`: final metrics for ten unchanged candidates, identifying the selected primary.
- `06_test_predictions.csv`: 1,600 source-matched Test predictions.
- `06_person_metrics.csv`: 40 model/person Test summaries.
- `06_cluster_bootstrap.csv`: 500 participant-cluster resamples for the selected primary.
- `06_primary_test_confusion.png`: held-out confusion matrix for Gaze Random Forest.

Final scoring uses the saved Stage-04/05 models and Train-fitted transformations; no Test refitting or model re-selection was performed.

In [8]:
# ======= 07. Create Evaluation Figures and Verify Stage-Owned Outputs ========

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ranking.sort_values('macro_f1').plot.barh(
    x='model', y='macro_f1', ax=axes[0], legend=False)
axes[0].set(xlabel='Macro F1', title='Validation model selection',
            xlim=(0, 1.03), ylabel='Candidate')
axes[1].bar(robustness.condition, robustness.macro_f1)
axes[1].set(xlabel='Fixed-feature perturbation',
            ylabel='Validation Macro F1', ylim=(0, 1.05),
            title='Early-fusion sensitivity (not attribution)')
axes[1].tick_params(axis='x', rotation=20)
fig.tight_layout()
fig.savefig(output_dir / '06_validation_evidence.png', dpi=170)
plt.show()
if RUN_FINAL_TEST:
    primary = prediction_table.loc[
        prediction_table.model.eq(selected_name)]
    fig, ax = plt.subplots(figsize=(4.6, 4.3))
    ConfusionMatrixDisplay.from_predictions(
        primary.target_index, primary.pred_index,
        labels=[0, 1], display_labels=['NR', 'TSR'],
        ax=ax, colorbar=False)
    ax.set_title('Primary held-out Test confusion')
    fig.tight_layout()
    fig.savefig(output_dir / '06_primary_test_confusion.png',
                dpi=170)
    plt.show()
    assert prediction_table.groupby('model').window_id.nunique().eq(
        len(test)).all()
    assert set(test_metrics.model) == set(MODEL_ORDER)
    print('Saved main Test evidence for',
          test.participant_id.nunique(), 'independent people.')
else:
    print('Final Test outputs are deferred until RUN_FINAL_TEST=True.')


Saved main Test evidence for 4 independent people.


**08. Scientific Conclusion**

The six-stage workflow now includes a **held-out Test evaluation** of the previously selected Gaze Random Forest on four new ZuCo 2.0 participants. The frozen model achieved **Test Macro F1 0.4997**, **Balanced Accuracy 0.5250** and **Accuracy 0.5250** on **160 windows**. Its Validation Macro F1 was **0.6244**.

**Primary finding:** The selected gaze forest's Test performance is lower than its Validation score. It correctly classified **60/80 NR** and **24/80 TSR** windows. The four participant-level Macro F1 values range from **0.3333 to 0.5238**, indicating uneven generalisation.

**Secondary comparison:** Equal-weight Late Fusion (**Test Macro F1 0.5204**) and Gated Fusion GRU (**0.5157**) score slightly above the frozen primary on Test. They remain *secondary observations*, not new selected winners, because Test was not available for model selection.

**Limits:** The Test has just **four independent participants**; confidence intervals are exploratory. Task/stimulus differences, unverified EEG–EyeLink clock drift and gaze calibration still restrict interpretation. This is **within-ZuCo task evaluation**, not external validation, a physiological explanation or a comprehension detector.

In [9]:
# =============== 08. Final Interpretation and Reproducibility ================

download_outputs_from_colab("eeg_eye_tracking_fusion/output", "06")


------------------------------------------------------------------------
Notebook 06
Files packaged: 4
  + 06_model_selection.csv
  + 06_validation_evidence.png
  + 06_validation_person_metrics.csv
  + 06_validation_sensitivity.csv
Download ZIP from Colab Contents: /content/eeg_eye_tracking_fusion_Notebook_06_Outputs.zip
